In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from pathlib import Path
from typing import List
from IPython.display import display

params = {
    'savefig.dpi': 300,
    'axes.labelsize': 22,
    'axes.titlesize': 26,
    'font.size': 22,
    'legend.fontsize': 20,
    'xtick.labelsize': 20,
    'ytick.labelsize': 20,
    'text.usetex': False,
    'figure.figsize': [6, 6],
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans']
}
plt.rcParams.update(params)
sns.set_theme(style="whitegrid")

In [ ]:
csv_path = Path("../data/benchmarks/urban/benchmark_results.csv")

plots_dir = Path("../data/benchmarks/urban/plots")
plots_dir.mkdir(parents=True, exist_ok=True)

print(f"Reading data from: {csv_path.resolve()}")
print(f"Saving plots to: {plots_dir.resolve()}\n")

# Load the benchmark data
df = pd.read_csv(csv_path)
display(df.head())

In [ ]:
def _window_label(subdf: pd.DataFrame) -> str:
    if "window_sec" not in subdf.columns:
        return ""
    vals = pd.to_numeric(subdf["window_sec"], errors="coerce").dropna().unique()
    vals = np.sort(vals)
    if vals.size == 0:
        return ""
    if vals.size == 1:
        return f"(window={vals[0]:g}s)"
    return f"(window in [{vals[0]:g}..{vals[-1]:g}]s, n={vals.size})"

def _window_suffix(subdf: pd.DataFrame) -> str:
    if "window_sec" not in subdf.columns:
        return ""
    vals = pd.to_numeric(subdf["window_sec"], errors="coerce").dropna().unique()
    vals = np.sort(vals)
    if vals.size == 1:
        return f"_W{vals[0]:g}s"
    if vals.size > 1:
        return "_Wmulti"
    return ""

def _add_ratio_top_axis(ax: plt.Axes, *, window_sec: float) -> None:
    if not np.isfinite(window_sec) or window_sec <= 0:
        return
    def sec_to_ratio(x): return np.asarray(x) / float(window_sec)
    def ratio_to_sec(r): return np.asarray(r) * float(window_sec)

    secax = ax.secondary_xaxis("top", functions=(sec_to_ratio, ratio_to_sec))
    secax.set_xlabel("lag/window ratio", labelpad=12)
    secax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=6))
    secax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.3g"))

def _group_by_window(d: pd.DataFrame) -> List[pd.DataFrame]:
    if "window_sec" not in d.columns: return [d]
    wvals = pd.to_numeric(d["window_sec"], errors="coerce").dropna().unique()
    return [d[d["window_sec"] == w].copy() for w in np.sort(wvals)] if wvals.size > 0 else [d]

def _require_cols(d: pd.DataFrame, cols: List[str]) -> bool:
    return all(c in d.columns for c in cols)

In [ ]:
# ==========================================
# 1. SCALING PLOTS 
# ==========================================
dsc_all = df[df["experiment"] == "scaling"].copy()

if not dsc_all.empty:
    for c in ["njobs", "wall_sec", "wall_p25", "wall_p75"]:
        if c in dsc_all.columns:
            dsc_all[c] = pd.to_numeric(dsc_all[c], errors="coerce")

    for gwin in _group_by_window(dsc_all):
        if gwin.empty: continue
        win_label = _window_label(gwin)
        suffix = _window_suffix(gwin)

        g2 = gwin.copy()
        g2["speedup"] = np.nan
        g2["efficiency"] = np.nan

        for mode in sorted(g2["mode"].astype(str).unique()):
            sub = g2[g2["mode"] == mode].sort_values("njobs").dropna(subset=["njobs", "wall_sec"])
            if sub.empty: continue
            p0, t0 = int(sub["njobs"].iloc[0]), float(sub["wall_sec"].iloc[0])

            for idx, row in sub.iterrows():
                p, tp = int(row["njobs"]), float(row["wall_sec"])
                S = (t0 / tp) if tp > 0 else np.nan
                g2.loc[idx, "speedup"] = S
                g2.loc[idx, "efficiency"] = S / (p / p0) if (p0 > 0 and p > 0) else np.nan

        # Plot 1A: Wall Time
        fig, ax = plt.subplots(layout="constrained")
        if _require_cols(gwin, ["mode", "njobs", "wall_sec", "wall_p25", "wall_p75"]):
            for mode, subm in gwin.groupby("mode"):
                subm = subm.sort_values("njobs").dropna(subset=["njobs", "wall_sec", "wall_p25", "wall_p75"])
                if subm.empty: continue
                x, y = subm["njobs"].to_numpy(), subm["wall_sec"].to_numpy()
                y_low = np.clip(y - subm["wall_p25"].to_numpy(), 0.0, None)
                y_high = np.clip(subm["wall_p75"].to_numpy() - y, 0.0, None)
                ax.errorbar(x, y, yerr=[y_low, y_high], fmt="o-", capsize=4, label=str(mode))
        else:
            sns.lineplot(data=gwin, x="njobs", y="wall_sec", hue="mode", marker="o", ax=ax)

        ax.set_title(f"Strong scaling: Wall time vs cores {win_label}".strip(), pad=20)
        ax.set_xlabel("njobs (processes)")
        ax.set_ylabel("wall time (s)")
        ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
        ax.legend(title="mode", frameon=True, loc="best")
        plt.savefig(plots_dir / f"scaling_wall_time{suffix}.png")
        plt.show() 

        # Plot 1B: Efficiency
        fig, ax = plt.subplots(layout="constrained")
        sns.lineplot(data=g2, x="njobs", y="efficiency", hue="mode", marker="o", ax=ax)
        ax.set_title(f"Strong scaling: Parallel efficiency {win_label}".strip(), pad=20)
        ax.set_xlabel("njobs (processes)")
        ax.set_ylabel("efficiency")
        ax.set_ylim(0.0, 1.05)
        ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
        ax.legend(title="mode", frameon=True, loc="best")
        plt.savefig(plots_dir / f"scaling_efficiency{suffix}.png")
        plt.show() 

# ==========================================
# 2. COMPLEXITY PLOTS
# ==========================================
dcp_all = df[df["experiment"] == "complexity"].copy()

if not dcp_all.empty:
    for c in ["max_lag_sec", "wall_sec", "wall_p25", "wall_p75", "window_sec"]:
        if c in dcp_all.columns:
            dcp_all[c] = pd.to_numeric(dcp_all[c], errors="coerce")

    for gwin in _group_by_window(dcp_all):
        if gwin.empty: continue
        win_label = _window_label(gwin)
        suffix = _window_suffix(gwin)
        w_unique = pd.to_numeric(gwin.get("window_sec", pd.Series(dtype=float)), errors="coerce").dropna().unique()
        window_for_axis = float(w_unique[0]) if w_unique.size == 1 else None

        # Plot 2A: Wall Time vs Lag
        fig, ax = plt.subplots(layout="constrained")
        if _require_cols(gwin, ["mode", "max_lag_sec", "wall_sec", "wall_p25", "wall_p75"]):
            for mode, subm in gwin.groupby("mode"):
                subm = subm.sort_values("max_lag_sec").dropna(subset=["max_lag_sec", "wall_sec", "wall_p25", "wall_p75"])
                if subm.empty: continue
                x, y = subm["max_lag_sec"].to_numpy(), subm["wall_sec"].to_numpy()
                y_low = np.clip(y - subm["wall_p25"].to_numpy(), 0.0, None)
                y_high = np.clip(subm["wall_p75"].to_numpy() - y, 0.0, None)
                ax.errorbar(x, y, yerr=[y_low, y_high], fmt="o-", capsize=4, label=str(mode))
        else:
            sns.lineplot(data=gwin, x="max_lag_sec", y="wall_sec", hue="mode", marker="o", ax=ax)
        
        ax.set_title(f"Complexity sweep: Wall time vs max lag {win_label}".strip(), pad=20)
        ax.set_xlabel("max_lag_sec (s)")
        ax.set_ylabel("wall time (s)")
        ax.set_ylim(bottom=0.0)
        ax.legend(title="mode", frameon=True, loc="best")
        if window_for_axis is not None: _add_ratio_top_axis(ax, window_sec=window_for_axis)
        plt.savefig(plots_dir / f"complexity_wall_time_vs_lag{suffix}.png")
        plt.show()

        # Plot 2B: Speedup vs Lag
        conv = gwin[gwin["mode"] == "conventional"][["max_lag_sec", "wall_sec"]].rename(columns={"wall_sec": "t_conv"})
        v1 = gwin[gwin["mode"] == "v1"][["max_lag_sec", "wall_sec"]].rename(columns={"wall_sec": "t_v1"})
        m = conv.merge(v1, on="max_lag_sec", how="inner").dropna(subset=["max_lag_sec", "t_conv", "t_v1"])
        
        if not m.empty:
            m = m.sort_values("max_lag_sec")
            m["speedup_conv_over_v1"] = m["t_conv"] / m["t_v1"]
            l_vals, s_vals = m["max_lag_sec"].to_numpy(), m["speedup_conv_over_v1"].to_numpy()

            x_cross = None
            if np.nanmin(s_vals) < 1.0 and np.nanmax(s_vals) > 1.0:
                x_cross = float(np.interp(1.0, s_vals[::-1], l_vals[::-1])) if s_vals[0] > s_vals[-1] else float(np.interp(1.0, s_vals, l_vals))

            fig, ax = plt.subplots(layout="constrained")
            sns.lineplot(data=m, x="max_lag_sec", y="speedup_conv_over_v1", marker="o", ax=ax)
            ax.axhline(1.0, color="red", linestyle="--", linewidth=1.5, label="crossover (y=1)")

            if x_cross is not None:
                ax.axvline(x_cross, color="gray", linestyle=":", alpha=0.8, linewidth=2)
                ratio_cross = (x_cross / window_for_axis) if (window_for_axis is not None and window_for_axis > 0) else np.nan
                label_text = f"Lag={x_cross:.2f}s\nRatio={ratio_cross:.3f}" if np.isfinite(ratio_cross) else f"Lag={x_cross:.2f}s"
                ax.annotate(label_text, xy=(x_cross, 1.0), xytext=(x_cross + 0.08 * float(np.nanmax(l_vals)), 1.3),
                            arrowprops=dict(arrowstyle="->", color="black", lw=1.2),
                            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="black", alpha=0.85))
                print(f"-> Crossover lag detected at {x_cross:.3g}s")

            ax.set_title(f"Speedup (conv / v1) vs max lag {win_label}".strip(), pad=20)
            ax.set_xlabel("max_lag_sec (s)")
            ax.set_ylabel("speedup (conv / v1)")
            ax.set_ylim(bottom=0.0)
            ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=7))
            ax.legend(frameon=True, loc="best")
            if window_for_axis is not None: _add_ratio_top_axis(ax, window_sec=window_for_axis)
            plt.savefig(plots_dir / f"complexity_speedup_vs_lag{suffix}.png")
            plt.show()

        # Plot 2C: Fidelity
        v1f = gwin[gwin["mode"] == "v1"].copy()
        
        if "rel_fro" in v1f.columns and v1f["rel_fro"].notna().any():
            fig, ax = plt.subplots(layout="constrained")
            sns.lineplot(data=v1f, x="max_lag_sec", y="rel_fro", marker="o", ax=ax)
            ax.set_title(f"Fidelity: rel_fro vs max lag {win_label}".strip(), pad=20)
            ax.set_xlabel("max_lag_sec (s)")
            ax.set_ylabel("rel_fro")
            ax.set_yscale("log")
            if window_for_axis is not None: _add_ratio_top_axis(ax, window_sec=window_for_axis)
            plt.savefig(plots_dir / f"fidelity_rel_fro_vs_lag{suffix}.png")
            plt.show()

        if "cos_p05" in v1f.columns and v1f["cos_p05"].notna().any():
            fig, ax = plt.subplots(layout="constrained")
            v1f["cos_error"] = 1.0 - v1f["cos_p05"]
            sns.lineplot(data=v1f, x="max_lag_sec", y="cos_error", marker="o", ax=ax)
            ax.set_title(f"Fidelity Error (1.0 - cos_p05) {win_label}".strip(), pad=20)
            ax.set_xlabel("max_lag_sec (s)")
            ax.set_ylabel("Error (1.0 - cos_p05)")
            ax.set_yscale("log")
            ax.yaxis.set_major_formatter(ticker.LogFormatterSciNotation())
            if window_for_axis is not None: _add_ratio_top_axis(ax, window_sec=window_for_axis)
            plt.savefig(plots_dir / f"fidelity_cos_p05_vs_lag{suffix}.png")
            plt.show()

print("\nAll plots generated and saved successfully!")